# P4 Null Model Confirmation

This notebook implements **Phase P4** from the frozen pipeline.

## Goals

- reconstruct the frozen row-normalized country projection for the count specification
- reconstruct the frozen row-normalized country projection for the capacity specification
- run the 40-replicate degree-preserving null model on each graph
- compute the null modularity distribution, z-score, and p-style summary
- confirm whether observed community structure is stronger than expected under rewired randomness

## Required outputs

- `artifacts/tables/p4_null_model_summary.csv`
- `artifacts/tables/p4_null_model_draws_count.csv`
- `artifacts/tables/p4_null_model_draws_capacity.csv`
- `artifacts/reports/p4_null_model_report.md`

Additional support output generated here:

- `artifacts/runtime/p4_null_model_manifest.json`

This notebook uses the frozen `P2` row-normalized matrices and the `P3` observed network summary as its inputs.


In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p2_country_mode_count_row_normalized.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P4__nogit'
SEED_BASE = 42
RESOLUTION = 1.0
TOP_K = 6
SIMILARITY = 'cosine'
NULL_RUNS = 40
EXPECTED_COUNTRIES = 94
EXPECTED_MODES = 16
EXPECTED_P3_MODULARITY = {
    'count': 0.6181083228635958,
    'capacity': 0.6843914198598924,
}
EXPECTED_P3_TOP5 = {
    'count': 0.083786,
    'capacity': 0.092553,
}

manifest = {
    'notebook': 'P4_Null_Model_Confirmation.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'inputs': {
        'count_row_normalized': str(TABLES / 'p2_country_mode_count_row_normalized.csv'),
        'capacity_row_normalized': str(TABLES / 'p2_country_mode_capacity_row_normalized.csv'),
        'p3_network_summary': str(TABLES / 'p3_country_projection_network_summary.csv'),
    },
    'frozen_rules': {
        'projection_similarity': SIMILARITY,
        'top_k': TOP_K,
        'community_method': 'networkx.community.louvain_communities',
        'community_seed': SEED_BASE,
        'resolution': RESOLUTION,
        'null_runs': NULL_RUNS,
        'null_rewire': 'nx.double_edge_swap',
        'weight_treatment': 'permute original edge weights onto rewired graph',
        'seed_schedule': '42 + i',
        'z_score_ddof': 0,
    },
}
(RUNTIME / 'p4_null_model_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Inputs:** `P2` row-normalized matrices + `P3` network summary'))


## Load Inputs and Verify the Observed Graph State

Before running the null model, the notebook checks that the observed row-normalized country projections still match the `P3` baseline. The null model should not silently drift away from the already frozen observed graph.


In [ ]:
count_row = pd.read_csv(TABLES / 'p2_country_mode_count_row_normalized.csv').set_index('Country/Area')
capacity_row = pd.read_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv').set_index('Country/Area')
p3_network = pd.read_csv(TABLES / 'p3_country_projection_network_summary.csv')

for name, matrix in {'count_row': count_row, 'capacity_row': capacity_row}.items():
    if matrix.shape != (EXPECTED_COUNTRIES, EXPECTED_MODES):
        raise ValueError(f'{name} shape drift: expected {(EXPECTED_COUNTRIES, EXPECTED_MODES)}, got {matrix.shape}')
    if matrix.isna().any().any():
        raise ValueError(f'{name} contains NaN values.')
    if (matrix < 0).any().any():
        raise ValueError(f'{name} contains negative values.')
    if not np.isclose(matrix.sum(axis=1).to_numpy(dtype=float), 1.0).all():
        raise ValueError(f'{name} is not row-normalized.')

display(pd.DataFrame({
    'matrix': ['count_row', 'capacity_row'],
    'shape': [count_row.shape, capacity_row.shape],
    'row_sum_min': [float(count_row.sum(axis=1).min()), float(capacity_row.sum(axis=1).min())],
    'row_sum_max': [float(count_row.sum(axis=1).max()), float(capacity_row.sum(axis=1).max())],
}))


## Helper Functions

The helper functions below reconstruct the frozen observed graph and implement the degree-preserving null model.


In [ ]:
def top_share(series, n=5):
    ordered = pd.Series(series, dtype=float).sort_values(ascending=False)
    total = float(ordered.sum())
    if total <= 0:
        return np.nan
    return float(ordered.head(n).sum() / total)


def build_projection_graph(matrix, similarity='cosine', top_k=6):
    names = list(matrix.index)
    values = matrix.to_numpy(dtype=float)
    graph = nx.Graph()
    graph.add_nodes_from(names)

    if len(names) < 2:
        return graph

    if similarity == 'cosine':
        sim = cosine_similarity(values)
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unknown similarity: {similarity}')

    np.fill_diagonal(sim, 0)
    k = min(top_k, len(names) - 1)

    for i, source in enumerate(names):
        neighbor_idx = np.argsort(sim[i])[-k:]
        for j in neighbor_idx:
            if i == j:
                continue
            weight = float(sim[i, j])
            if weight <= 0:
                continue
            target = names[j]
            if graph.has_edge(source, target):
                graph[source][target]['weight'] = max(graph[source][target]['weight'], weight)
            else:
                graph.add_edge(source, target, weight=weight)

    return graph


def detect_communities(graph, seed=42, resolution=1.0):
    if graph.number_of_nodes() == 0:
        return pd.Series(dtype=int), [], np.nan, pd.Series(dtype=float), pd.Series(dtype=float)

    if graph.number_of_edges() == 0:
        assignment = pd.Series({node: idx + 1 for idx, node in enumerate(sorted(graph.nodes()))}, name='community')
        communities = [{node} for node in assignment.index]
        strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
        degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
        return assignment, communities, 0.0, strengths, degrees

    communities = list(nx.community.louvain_communities(graph, weight='weight', seed=seed, resolution=resolution))
    communities = sorted(communities, key=len, reverse=True)
    assignment = pd.Series(
        {node: cid for cid, members in enumerate(communities, start=1) for node in members},
        name='community',
    ).sort_index()
    strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
    degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
    modularity = float(nx.community.modularity(graph, communities, weight='weight'))
    return assignment, communities, modularity, strengths, degrees


def run_null_modularity_test(graph, observed_modularity, weight_kind, runs=40, base_seed=42):
    if graph.number_of_edges() < 4:
        empty = pd.DataFrame(columns=['replicate_id', 'seed', 'rewire_success', 'degree_sequence_preserved', 'null_modularity'])
        summary = {
            'weight_kind': weight_kind,
            'observed_modularity': observed_modularity,
            'null_mean': np.nan,
            'null_std': np.nan,
            'null_min': np.nan,
            'null_median': np.nan,
            'null_max': np.nan,
            'observed_minus_null_mean': np.nan,
            'z_score': np.nan,
            'null_p_ge_observed': np.nan,
            'null_runs': 0,
            'passes_null_gate': False,
        }
        return empty, summary

    edge_list = list(graph.edges())
    edge_weights = np.array([graph[u][v]['weight'] for u, v in edge_list], dtype=float)
    original_degree_sequence = sorted(dict(graph.degree()).values())
    draw_rows = []

    for replicate_id in range(runs):
        run_seed = base_seed + replicate_id
        rewired = nx.Graph()
        rewired.add_nodes_from(graph.nodes())
        rewired.add_edges_from(edge_list)
        nswap = max(10, 5 * rewired.number_of_edges())
        max_tries = max(100, 40 * rewired.number_of_edges())

        nx.double_edge_swap(rewired, nswap=nswap, max_tries=max_tries, seed=run_seed)
        degree_sequence_preserved = sorted(dict(rewired.degree()).values()) == original_degree_sequence
        if not degree_sequence_preserved:
            raise ValueError(f'{weight_kind}: degree sequence drift detected in null replicate {replicate_id}.')

        shuffled_weights = np.random.default_rng(run_seed).permutation(edge_weights)
        for (u, v), weight in zip(rewired.edges(), shuffled_weights):
            rewired[u][v]['weight'] = float(weight)

        _, _, null_modularity, _, _ = detect_communities(rewired, seed=run_seed, resolution=RESOLUTION)
        draw_rows.append({
            'replicate_id': int(replicate_id),
            'seed': int(run_seed),
            'rewire_success': True,
            'degree_sequence_preserved': True,
            'null_modularity': float(null_modularity),
        })

    draws = pd.DataFrame(draw_rows)
    null_mean = float(draws['null_modularity'].mean())
    null_std = float(draws['null_modularity'].std(ddof=0))
    z_score = (observed_modularity - null_mean) / null_std if pd.notna(null_std) and null_std > 0 else np.nan
    p_ge = float((draws['null_modularity'] >= observed_modularity).mean())
    summary = {
        'weight_kind': weight_kind,
        'observed_modularity': float(observed_modularity),
        'null_mean': null_mean,
        'null_std': null_std,
        'null_min': float(draws['null_modularity'].min()),
        'null_median': float(draws['null_modularity'].median()),
        'null_max': float(draws['null_modularity'].max()),
        'observed_minus_null_mean': float(observed_modularity - null_mean),
        'z_score': float(z_score) if pd.notna(z_score) else np.nan,
        'null_p_ge_observed': p_ge,
        'null_runs': int(len(draws)),
        'passes_null_gate': bool((pd.notna(z_score) and z_score >= 2.0) or (pd.notna(p_ge) and p_ge <= 0.05)),
    }
    return draws, summary


## Reconstruct the Observed Graphs and Run the Null Model

This section verifies that the observed graphs match the `P3` baseline and then runs the frozen 40-replicate null model separately for count and capacity.


In [ ]:
observed_objects = {}
summary_rows = []

for weight_kind, matrix in [('count', count_row), ('capacity', capacity_row)]:
    graph = build_projection_graph(matrix, similarity=SIMILARITY, top_k=TOP_K)
    assignment, communities, observed_modularity, strengths, degrees = detect_communities(graph, seed=SEED_BASE, resolution=RESOLUTION)
    observed_top5 = round(top_share(strengths, 5), 6)

    p3_row = p3_network.loc[p3_network['weight_kind'] == weight_kind].iloc[0]
    if graph.number_of_nodes() != EXPECTED_COUNTRIES:
        raise ValueError(f'{weight_kind}: observed graph lost countries.')
    if int(p3_row['nodes']) != graph.number_of_nodes() or int(p3_row['edges']) != graph.number_of_edges():
        raise ValueError(f'{weight_kind}: observed graph topology drifted from P3.')
    if not math.isclose(float(p3_row['modularity']), observed_modularity, rel_tol=0, abs_tol=1e-12):
        raise ValueError(f'{weight_kind}: observed modularity drifted from P3.')
    if not math.isclose(observed_modularity, EXPECTED_P3_MODULARITY[weight_kind], rel_tol=0, abs_tol=1e-12):
        raise ValueError(f'{weight_kind}: observed modularity drifted from the frozen baseline constant.')
    if not math.isclose(observed_top5, EXPECTED_P3_TOP5[weight_kind], rel_tol=0, abs_tol=1e-6):
        raise ValueError(f'{weight_kind}: observed top-5 share drifted from P3/P2.')

    draws, summary = run_null_modularity_test(
        graph,
        observed_modularity=observed_modularity,
        weight_kind=weight_kind,
        runs=NULL_RUNS,
        base_seed=SEED_BASE,
    )

    if len(draws) != NULL_RUNS:
        raise ValueError(f'{weight_kind}: null draw count mismatch.')
    if not draws['rewire_success'].all() or not draws['degree_sequence_preserved'].all():
        raise ValueError(f'{weight_kind}: one or more null replicates failed rewiring QA.')

    draws_path = TABLES / f'p4_null_model_draws_{weight_kind}.csv'
    draws.to_csv(draws_path, index=False)

    summary.update({
        'observed_nodes': int(graph.number_of_nodes()),
        'observed_edges': int(graph.number_of_edges()),
        'observed_communities': int(len(communities)),
        'observed_top5_strength_share': observed_top5,
    })
    summary_rows.append(summary)

    observed_objects[weight_kind] = {
        'graph': graph,
        'assignment': assignment,
        'strengths': strengths,
        'degrees': degrees,
        'draws': draws,
    }

null_summary = pd.DataFrame(summary_rows).sort_values('weight_kind').reset_index(drop=True)
null_summary.to_csv(TABLES / 'p4_null_model_summary.csv', index=False)

display(null_summary)


## Write the P4 Report

The report below records the frozen null-model configuration and the observed-vs-null comparison for both analytical paths.


In [ ]:
report_lines = [
    '# P4 Null Model Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    '- inputs: `P2` row-normalized matrices checked against `P3` observed network summary',
    '',
    '## Frozen null-model rules used',
    f'- observed graph: row-based country projection using `{SIMILARITY}` similarity and `top_k={TOP_K}`',
    f'- observed community method: `networkx.community.louvain_communities` with `seed={SEED_BASE}` and `resolution={RESOLUTION}`',
    f'- null rewire routine: `nx.double_edge_swap` with `runs={NULL_RUNS}`',
    '- weight treatment: permute original edge weights onto each rewired graph',
    '- seed schedule: `42 + i` for rewiring, weight permutation, and null Louvain detection',
    '- z-score rule: `(observed_modularity - null_mean) / null_std` with `ddof=0`',
    '- p-style summary: share of null draws with `null_modularity >= observed_modularity`',
    '',
    '## Observed graph consistency check',
    '- observed graph topology and modularity were rechecked against `P3` before running the null model',
    '',
    '## Null-model summary',
]
for row in null_summary.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: observed={row.observed_modularity:.6f}, null_mean={row.null_mean:.6f}, null_std={row.null_std:.6f}, z={row.z_score:.6f}, p_ge_observed={row.null_p_ge_observed:.6f}, passes_null_gate={bool(row.passes_null_gate)}'
    )

report_lines.extend([
    '',
    '## Interpretation boundary',
    '- Null-model results support non-random structural differentiation only.',
    '- They do not establish causal mechanism.',
    '',
    '## Output inventory',
    '- `artifacts/tables/p4_null_model_summary.csv`',
    '- `artifacts/tables/p4_null_model_draws_count.csv`',
    '- `artifacts/tables/p4_null_model_draws_capacity.csv`',
    '- `artifacts/reports/p4_null_model_report.md`',
    '- `artifacts/runtime/p4_null_model_manifest.json`',
])

(REPORTS / 'p4_null_model_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
display(Markdown((REPORTS / 'p4_null_model_report.md').read_text(encoding='utf-8')))
